In [4]:

from dotenv import load_dotenv
load_dotenv()

True

In [12]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0)
response = llm.invoke("what is AI? Tell me in one sentence")
response.content

'AI is a branch of computer science that enables machines to simulate human intelligence, allowing them to learn, reason, and perform tasks that typically require human cognition.'

In [13]:
from langchain_core.documents import Document
my_text="""Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.[1]

High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g., text, images, audio, and videos).

The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach these goals, AI researchers use techniques including state space search and mathematical optimisation, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics.[b] AI also draws upon psychology, linguistics, philosophy, neuroscience, and other fields.[2] Some companies, such as OpenAI, Google DeepMind, and Meta, aim to create artificial general intelligence (AGI)—AI that can complete nearly any cognitive task at least as well as a human.[3]"""

docs = [
    Document(
        page_content=my_text,
        metadata={"source": "wikipedia", "Document_id": "1"},
    )]

docs



[Document(metadata={'source': 'wikipedia', 'Document_id': '1'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.[1]\n\nHigh-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g., text, images, audio, and videos).\n\nThe traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotic

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)
chunks

[Document(metadata={'source': 'wikipedia', 'Document_id': '1'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.[1]'),
 Document(metadata={'source': 'wikipedia', 'Document_id': '1'}, page_content='High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g., text, images, audio, and videos).'),
 Document(metadata={'source': 'wikipedia', 'Document_id': '1'}, page_content='The traditional goals

In [15]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 658.07it/s]


In [16]:
from langchain_chroma import Chroma
db = Chroma.from_documents(chunks, embeddings)

In [17]:
from rag_learning.rag import retrieve

# similarity_search returns whatever is nearest - relevant or not, duplicates
# included. retrieve() keeps only distinct passages inside the relevance cutoff.
context = retrieve(db, 'what is AI?', k=3)
context

In [ ]:
from rag_learning.rag import answer_from_context

# Was: llm.invoke(f'... you can answer the question using this context: {context}')
# That phrasing invited the model to answer from its training data. This forbids
# it, and refuses without calling the model when nothing relevant was retrieved.
result = answer_from_context(db, llm, 'What is AI?', k=3)
print(result.answer)
print('grounded on:', result.sources)

In [ ]:
# Nothing in this four-chunk index is about France - the nearest chunk sits at
# distance 1.86, past the cutoff - so the model is never called. The refusal is
# fixed text, not a generation, which is the only answer that cannot be made up.
off_topic = answer_from_context(db, llm, 'What is the capital of France?')
print(off_topic.answer)
print('used retrieval:', off_topic.used_retrieval)